# Cleaning and Preparation of Prime2 Road Network Data

This section outlines the processing of the Prime2 national road network dataset to create a walkable network for accessibility analysis. The workflow involves filtering and cleaning the Prime2 road features, retaining only relevant attributes, and classifying road segments based on their functional type and pedestrian accessibility. Walking speeds are assigned to each road segment, and travel times are calculated to facilitate pedestrian network modeling. The resulting cleaned and enriched dataset is used to construct the walk and public transport network for subsequent accessibility analysis.

In [34]:
#imports required libraries
import geopandas as gpd

In [35]:
#Reads Way_GDF2_DCC shapefile
Way_gdf2 = gpd.read_file(r"D:\GIS-TU Dublin\Year_2\THESIS\DCC_Analysis\Data_used\Prime2_Data\Way_GDF2_DCC.shp")


In [36]:
#Shows all the information related to way_gdf2 prime2 data
Way_gdf2.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 15344 entries, 0 to 15343
Data columns (total 86 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   END_PNT_GU  15344 non-null  object  
 1   GUID        15344 non-null  object  
 2   START_PNT_  15344 non-null  object  
 3   URI         0 non-null      object  
 4   ACCESS_CAP  15344 non-null  object  
 5   ACCESS_CHA  15344 non-null  object  
 6   ACCESS_VAL  15344 non-null  object  
 7   DIRECTION_  15344 non-null  object  
 8   DIRECTION1  15344 non-null  object  
 9   DIRECTIO_1  15344 non-null  object  
 10  FORM_CAPTU  15344 non-null  object  
 11  FORM_CHANG  15344 non-null  object  
 12  FORM_VALID  15344 non-null  object  
 13  FUNC_CAPTU  15344 non-null  object  
 14  FUNC_CHANG  15344 non-null  object  
 15  FUNC_VALID  15344 non-null  object  
 16  LINE_GEOM_  15344 non-null  object  
 17  LINE_GEOM1  15344 non-null  object  
 18  LINE_GEO_1  15344 non-null  object  
 

In [71]:
#shows first 2 rows of dataframe
Way_gdf2.head(2)

,END_PNT_GU,GUID,START_PNT_,URI,ACCESS_CAP,ACCESS_CHA,ACCESS_VAL,DIRECTION_,DIRECTION1,DIRECTIO_1,FORM_CAPTU,FORM_CHANG,FORM_VALID,FUNC_CAPTU,FUNC_CHANG,FUNC_VALID,LINE_GEOM_,LINE_GEOM1,LINE_GEO_1,MAX_AXLES_,MAX_AXLES1,MAX_AXLE_1,MAX_HT_CAP,MAX_HT_CHA,MAX_HT_VAL,MAX_WD_CAP,MAX_WD_CHA,MAX_WD_VAL,MAX_WT_CAP,MAX_WT_CHA,MAX_WT_VAL,STATUS_CAP,STATUS_CHA,STATUS_VAL,Z_ORDER_CA,Z_ORDER_CH,Z_ORDER_VA,Z_ORDER__1,Z_ORDER__2,Z_ORDER__3,...,MAX_WT_V_2,MAX_WT_C_1,MAX_WT_C_2,MAX_WD_V_1,MAX_WD_V_2,MAX_WD_C_1,MAX_WD_C_2,MAX_HT_V_1,MAX_HT_V_2,MAX_HT_C_1,MAX_HT_C_2,MAX_AXLE_2,MAX_AXLE_3,MAX_AXLE_4,MAX_AXLE_5,LINE_GEO_2,LINE_GEO_3,LINE_GEO_4,LINE_GEO_5,LINE_GEO_6,LINE_GEO_7,FUNC_VAL_1,FUNC_ID,FUNC_CHA_1,FUNC_CAP_1,FORM_VAL_1,FORM_ID,FORM_CHA_1,ACCESS_C_1,CAPTURE_SP,DIRECTIO_2,DIRECTIO_3,FORM_CAP_1,DIRECTIO_4,DIRECTIO_5,ACCESS_V_1,ACCESS_ID,ACCESS_C_2,Shape_Leng,geometry
0,403f686e-05d5-4302-8632-76ab7ac899fa,0f18bd8f-5441-48ff-8343-95f250df061f,dc3c4114-357f-46ef-b91a-2d14e6148aa7,None,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-10,2012-01-10,2012-01-01,None,None,None,None,None,None,None,None,None,None,None,None,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,0,1,7,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,7,1,1,0,1,409,7,1,1,362,7,1,1,1,1,1,7,1,1,1,7,249.577182,"LINESTRING Z (721845.332 738194.297 0.000, 721..."
1,3b008afe-bcb4-4035-b158-b956934508fd,fa145cf4-b979-4bfa-a07b-f5c250977573,836ca3e8-b6b5-4414-b1d5-1638c8bc9400,None,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-10,2012-01-10,2012-01-01,None,None,None,None,None,None,None,None,None,None,None,None,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,2012-01-01,0,1,7,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,7,1,1,0,1,191,7,1,1,362,7,1,1,1,1,1,7,1,1,1,7,36.261145,"LINESTRING Z (722756.985 739914.019 0.000, 722..."


In [72]:
#Only keeps the required column
keep = [
    # ids & geometry
    "GUID", "START_PNT_", "END_PNT_GU", "geometry",
    # length
    "Shape_Leng",
    # form/functional/access/direction
    "FORM_VAL_1", "FORM_ID",
    "FUNC_VAL_1", "FUNC_ID",
    "ACCESS_VAL", "ACCESS_ID",
    "DIRECTION_", "DIRECTIO_2",
    # z-order (keep just one)
    "Z_ORDER__4"
]
keep = [c for c in keep if c in Way_gdf2.columns]

Way_gdf2_clean = Way_gdf2[keep].copy()

Way_gdf2_clean.head()

,GUID,START_PNT_,END_PNT_GU,geometry,Shape_Leng,FORM_VAL_1,FORM_ID,FUNC_VAL_1,FUNC_ID,ACCESS_VAL,ACCESS_ID,DIRECTION_,DIRECTIO_2,Z_ORDER__4
0,0f18bd8f-5441-48ff-8343-95f250df061f,dc3c4114-357f-46ef-b91a-2d14e6148aa7,403f686e-05d5-4302-8632-76ab7ac899fa,"LINESTRING Z (721845.332 738194.297 0.000, 721...",249.577182,1,362,1,409,2012-01-01,1,2012-01-01,1,1
1,fa145cf4-b979-4bfa-a07b-f5c250977573,836ca3e8-b6b5-4414-b1d5-1638c8bc9400,3b008afe-bcb4-4035-b158-b956934508fd,"LINESTRING Z (722756.985 739914.019 0.000, 722...",36.261145,1,362,1,191,2012-01-01,1,2012-01-01,1,1
2,aa260a66-0d2b-466a-8cf7-fe41cb5191b3,6187ff32-4e1f-40a0-9843-aee825a5448b,1f410ec7-6f4a-4ed8-b17b-b1d1617fa0fe,"LINESTRING Z (720792.505 738987.431 0.000, 720...",83.084423,1,362,1,667,2012-01-01,1,2012-01-01,1,1
3,397739b3-4c5f-40e2-a461-7f9c4eab6724,a3ca0141-e653-4854-a775-f90ae809c7a1,be6c36bb-da06-4172-8f89-0020c5fe94e5,"LINESTRING Z (721247.613 739594.899 0.000, 721...",80.866832,1,362,1,191,2012-01-01,1,2012-01-01,1,1
4,9ca50adc-6481-4682-a7f2-7426d1db8b97,21bd3d01-d3bb-40fd-910f-48b1377a6ec1,523c746e-0a04-4a2b-9633-fc12fc094cef,"LINESTRING Z (720201.500 738847.934 0.000, 720...",116.396436,1,362,1,191,2012-01-01,1,2012-01-01,1,1


In [73]:
#Shows the unique values of column 'FORM_VAL_1'
Way_gdf2_clean['FORM_VAL_1'].value_counts()

1    15344
Name: FORM_VAL_1, dtype: int64

In [74]:
#Shows the unique values of column 'FORM_VAL_1'
Way_gdf2_clean['FORM_VAL_1'].value_counts()

1    15344
Name: FORM_VAL_1, dtype: int64

In [75]:
#Shows the unique values of column 'FUNC_ID'
Way_gdf2_clean['FUNC_ID'].value_counts()

191    9569
475    2176
409    2103
667    1233
177     130
266      84
660      26
659      20
662       2
661       1
Name: FUNC_ID, dtype: int64

In [76]:
#Shows the unique values of column 'ACCESS_ID'
Way_gdf2_clean['ACCESS_ID'].value_counts()

1    15259
5       69
4       15
7        1
Name: ACCESS_ID, dtype: int64

In [77]:
#Shows the unique values of column 'FORM_ID'
Way_gdf2_clean['FORM_ID'].value_counts()

362    15060
122      196
246       37
369       23
281       21
215        5
411        1
655        1
Name: FORM_ID, dtype: int64

In [78]:
#Shows the unique values of column 'Z_ORDER__4'
Way_gdf2_clean['Z_ORDER__4'].value_counts()

1    15344
Name: Z_ORDER__4, dtype: int64

In [90]:
 #--- Define lookup dicts ---
road_class_map = {
    667: "First Class (Motorway)",
    191: "Second Class",
    409: "Third Class (Access Only)",
    475: "Fourth Class",
    177: "Fifth Class",
    266: "Sixth Class",
    659: "Pedestrian Route",
    660: "Cycleway",
    661: "Slip Road",
    662: "Service Link"
}

speed_map = {
    667: 120,   # First Class
    191: 100,   # Second Class
    409: 80,    # Third Class
    475: 80,    # Fourth Class
    177: 50,    # Fifth Class (urban)
    266: 50,    # Sixth Class
    659: 3.6,   # Pedestrian Route (elderly, 1.0 m/s)
    660: 15,    # Cycleway
    661: 60,    # Slip Road
    662: 30     # Service Link
}

form_lookup = {
    362: "Single Carriageway",
    122: "Dual Carriageway",
    246: "Roundabout",
    369: "Slip Road",
    281: "Track / Local Lane",
    215: "Footpath",
    411: "Cycleway",
    655: "Service Road"
}

access_map = {
    1: "All Traffic",
    5: "Pedestrian Only",
    4: "Pedestrian & Cycle",
    7: "No Pedestrian Access"
}


# --- Create new fields ---
Way_gdf2_clean["ROAD_CLASS"] = Way_gdf2_clean["FUNC_ID"].map(road_class_map).fillna("Unknown")
Way_gdf2_clean["FormClass"] = Way_gdf2_clean["FORM_ID"].map(form_lookup).fillna("Unknown")
Way_gdf2_clean["AccessClass"] = Way_gdf2_clean["ACCESS_ID"].map(access_map).fillna("Unknown")
Way_gdf2_clean["Speed_kmh"]  = Way_gdf2_clean["FUNC_ID"].map(speed_map).fillna(50)  # fallback = 50 km/h
Way_gdf2_clean["Time"] = ((Way_gdf2_clean["Shape_Leng"] / 1000) / Way_gdf2_clean["Speed_kmh"] * 60).round(2)

In [91]:
#Displays first few rows
Way_gdf2_clean.head()

,GUID,START_PNT_,END_PNT_GU,geometry,Shape_Leng,FORM_VAL_1,FORM_ID,FUNC_VAL_1,FUNC_ID,ACCESS_VAL,ACCESS_ID,DIRECTION_,DIRECTIO_2,Z_ORDER__4,ROAD_CLASS,FormClass,AccessClass,Speed_kmh,Time
0,0f18bd8f-5441-48ff-8343-95f250df061f,dc3c4114-357f-46ef-b91a-2d14e6148aa7,403f686e-05d5-4302-8632-76ab7ac899fa,"LINESTRING Z (721845.332 738194.297 0.000, 721...",249.577182,1,362,1,409,2012-01-01,1,2012-01-01,1,1,Third Class (Access Only),Single Carriageway,All Traffic,80.0,0.19
1,fa145cf4-b979-4bfa-a07b-f5c250977573,836ca3e8-b6b5-4414-b1d5-1638c8bc9400,3b008afe-bcb4-4035-b158-b956934508fd,"LINESTRING Z (722756.985 739914.019 0.000, 722...",36.261145,1,362,1,191,2012-01-01,1,2012-01-01,1,1,Second Class,Single Carriageway,All Traffic,100.0,0.02
2,aa260a66-0d2b-466a-8cf7-fe41cb5191b3,6187ff32-4e1f-40a0-9843-aee825a5448b,1f410ec7-6f4a-4ed8-b17b-b1d1617fa0fe,"LINESTRING Z (720792.505 738987.431 0.000, 720...",83.084423,1,362,1,667,2012-01-01,1,2012-01-01,1,1,First Class (Motorway),Single Carriageway,All Traffic,120.0,0.04
3,397739b3-4c5f-40e2-a461-7f9c4eab6724,a3ca0141-e653-4854-a775-f90ae809c7a1,be6c36bb-da06-4172-8f89-0020c5fe94e5,"LINESTRING Z (721247.613 739594.899 0.000, 721...",80.866832,1,362,1,191,2012-01-01,1,2012-01-01,1,1,Second Class,Single Carriageway,All Traffic,100.0,0.05
4,9ca50adc-6481-4682-a7f2-7426d1db8b97,21bd3d01-d3bb-40fd-910f-48b1377a6ec1,523c746e-0a04-4a2b-9633-fc12fc094cef,"LINESTRING Z (720201.500 738847.934 0.000, 720...",116.396436,1,362,1,191,2012-01-01,1,2012-01-01,1,1,Second Class,Single Carriageway,All Traffic,100.0,0.07


In [92]:
#Checks CRS of dataframe
Way_gdf2_clean.crs

<Projected CRS: EPSG:2157>
Name: IRENET95 / Irish Transverse Mercator
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Ireland - onshore. United Kingdom (UK) - Northern Ireland (Ulster) - onshore.
- bounds: (-10.56, 51.39, -5.34, 55.43)
Coordinate Operation:
- name: Irish Transverse Mercator
- method: Transverse Mercator
Datum: IRENET95
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich


In [104]:
#Displays first few rows
Way_gdf2_clean.head(2)

,GUID,START_PNT_,END_PNT_GU,geometry,Shape_Leng,FORM_VAL_1,FORM_ID,FUNC_VAL_1,FUNC_ID,ACCESS_VAL,ACCESS_ID,DIRECTION_,DIRECTIO_2,Z_ORDER__4,ROAD_CLASS,FormClass,AccessClass,Speed_kmh,Time
0,0f18bd8f-5441-48ff-8343-95f250df061f,dc3c4114-357f-46ef-b91a-2d14e6148aa7,403f686e-05d5-4302-8632-76ab7ac899fa,"LINESTRING Z (721845.332 738194.297 0.000, 721...",249.577182,1,362,1,409,2012-01-01,1,2012-01-01,1,1,Third Class (Access Only),Single Carriageway,All Traffic,80.0,0.19
1,fa145cf4-b979-4bfa-a07b-f5c250977573,836ca3e8-b6b5-4414-b1d5-1638c8bc9400,3b008afe-bcb4-4035-b158-b956934508fd,"LINESTRING Z (722756.985 739914.019 0.000, 722...",36.261145,1,362,1,191,2012-01-01,1,2012-01-01,1,1,Second Class,Single Carriageway,All Traffic,100.0,0.02


In [106]:
# Save walk_gdf as shapefile
Way_gdf2_clean.to_file(r"D:\GIS-TU Dublin\Year_2\THESIS\DCC_Analysis\Data_used\Prime2_Data\transit_network.shp", driver="ESRI Shapefile")


In [94]:
# Display a summary of the GeoDataFrame, including column names, data types, and non-null counts
Way_gdf2_clean.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 15344 entries, 0 to 15343
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   GUID         15344 non-null  object  
 1   START_PNT_   15344 non-null  object  
 2   END_PNT_GU   15344 non-null  object  
 3   geometry     15344 non-null  geometry
 4   Shape_Leng   15344 non-null  float64 
 5   FORM_VAL_1   15344 non-null  int64   
 6   FORM_ID      15344 non-null  int64   
 7   FUNC_VAL_1   15344 non-null  int64   
 8   FUNC_ID      15344 non-null  int64   
 9   ACCESS_VAL   15344 non-null  object  
 10  ACCESS_ID    15344 non-null  int64   
 11  DIRECTION_   15344 non-null  object  
 12  DIRECTIO_2   15344 non-null  int64   
 13  Z_ORDER__4   15344 non-null  int64   
 14  ROAD_CLASS   15344 non-null  object  
 15  FormClass    15344 non-null  object  
 16  AccessClass  15344 non-null  object  
 17  Speed_kmh    15344 non-null  float64 
 18  Time         15344

In [98]:
# Step 1: Drop motorways and major national roads for creating walk network
walk_gdf = Way_gdf2_clean[~Way_gdf2_clean["FUNC_ID"].isin([667])].copy()

print("Before:", len(Way_gdf2_clean))
print("After removing motorways/nationals:", len(walk_gdf))

Before: 15344
After removing motorways/nationals: 14111


In [99]:
# Step 2: Keep only pedestrian-allowed links
walk_gdf = walk_gdf[walk_gdf["ACCESS_ID"].isin([1, 4, 5])].copy()

print("After filtering pedestrian access:", len(walk_gdf))
print(walk_gdf["ACCESS_ID"].value_counts())

After filtering pedestrian access: 14110
1    14026
5       69
4       15
Name: ACCESS_ID, dtype: int64


In [102]:
# Step 3: Assign elderly walking speed and recompute time (minutes)

# 1 m/s = 3.6 km/h
WALK_KMH = 3.6
walk_gdf["Speed_kmh"] = WALK_KMH
# Time (minutes) = (distance km / speed km/h) * 60
walk_gdf["Time"] = ((walk_gdf["Shape_Leng"] / 1000) / walk_gdf["Speed_kmh"] * 60).round(2)
walk_gdf.head(2)

,GUID,START_PNT_,END_PNT_GU,geometry,Shape_Leng,FORM_VAL_1,FORM_ID,FUNC_VAL_1,FUNC_ID,ACCESS_VAL,ACCESS_ID,DIRECTION_,DIRECTIO_2,Z_ORDER__4,ROAD_CLASS,FormClass,AccessClass,Speed_kmh,Time
0,0f18bd8f-5441-48ff-8343-95f250df061f,dc3c4114-357f-46ef-b91a-2d14e6148aa7,403f686e-05d5-4302-8632-76ab7ac899fa,"LINESTRING Z (721845.332 738194.297 0.000, 721...",249.577182,1,362,1,409,2012-01-01,1,2012-01-01,1,1,Third Class (Access Only),Single Carriageway,All Traffic,3.6,4.16
1,fa145cf4-b979-4bfa-a07b-f5c250977573,836ca3e8-b6b5-4414-b1d5-1638c8bc9400,3b008afe-bcb4-4035-b158-b956934508fd,"LINESTRING Z (722756.985 739914.019 0.000, 722...",36.261145,1,362,1,191,2012-01-01,1,2012-01-01,1,1,Second Class,Single Carriageway,All Traffic,3.6,0.60


In [103]:
# Save walk_gdf as shapefile
walk_gdf.to_file("D:\GIS-TU Dublin\Year_2\THESIS\DCC_Analysis\Data_used\Prime2_Data\walk_network.shp", driver="ESRI Shapefile")


[103]:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
